# Spatial comparison: SNOTEL, iSnobal, Noah-MP, and ASO — clean version

This notebook produces publication figures for both **snow water equivalent (SWE)** and **snow depth (SD)**.

Key decisions:

- Plot exactly four water years selected from the **East River watershed annual peak iSnobal SWE ranking**:
  **Low WY2018**, **Typical WY2022**, **High WY2019**, and **Abnormal WY2026**.
- The four selected years come from the previously completed watershed peak-SWE ranking; this notebook uses those selected years directly and does **not** recompute the ranking.
- Columns are ordered **Low → Typical → High → Abnormal**.
- Rows are **Butte**, **Schofield Pass**, and **Upper Taylor**.
- Each of the 12 panels is labeled sequentially **(a)–(l)** in row-major order.
- Plot SNOTEL, nearest iSnobal, nearest Noah-MP, the existing iSnobal min-max envelope inside the nearest Noah-MP cell, and all available ASO observations in the plotted years.
- ASO is sampled from the **single nearest raw ASO raster pixel center** at each SNOTEL station. No ASO QC, spatial averaging, interpolation, or regridding is applied. Source nodata remains missing.
- Legend labels are dataset names only: **SNOTEL**, **iSnobal**, **Noah-MP**, **ASO**.
- ASO observations use the established large-star symbol convention.
- No statistics or ASO differences are written on the plotted panels.
- For **WY2018-WY2026**, save SNOTEL-reference `r` and `MB` for iSnobal and Noah-MP, separately for SWE and SD.
- For every available ASO flight date in WY2018-WY2026, save `|dataset - ASO|` for SNOTEL, iSnobal, and Noah-MP, separately for SWE and SD.
- Export each figure as both **600-dpi PNG** and **vector PDF**.


## 1. Imports and project paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import sys

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Absolute East River v0.8.2 paths used by the existing project notebook.
# -------------------------------------------------------------------------
REPO_ROOT = Path(
    "/bsuscratch/zihaozhang/iSnobal/East_River_Workflow_Code_v0.8.2"
)
CONFIG_PATH = REPO_ROOT / "config" / "east_river_config.yaml"

# Allow the notebook to work even if the package was not installed editable.
try:
    from east_river_workflow.config import load_config
    from east_river_workflow.constants import MODEL_COLORS
    from east_river_workflow.data_access import read_aso_raster
    from east_river_workflow.grids import nearest_grid_cells
    from east_river_workflow.metrics import pearson_mean_bias
    from east_river_workflow.utils import water_year, water_year_bounds
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    from east_river_workflow.config import load_config
    from east_river_workflow.constants import MODEL_COLORS
    from east_river_workflow.data_access import read_aso_raster
    from east_river_workflow.grids import nearest_grid_cells
    from east_river_workflow.metrics import pearson_mean_bias
    from east_river_workflow.utils import water_year, water_year_bounds


## 2. Publication settings and requested case years

In [ ]:
# -------------------------------------------------------------------------
# Requested station rows and four plotted case-year columns.
# -------------------------------------------------------------------------
STATIONS = [
    (380, "Butte"),
    (737, "Schofield Pass"),
    (1141, "Upper Taylor"),
]

# Selected by the watershed annual peak iSnobal SWE ranking:
#   Low      = WY2018  (244.0 mm)
#   Typical  = WY2022  (344.9 mm)
#   High     = WY2019  (497.7 mm)
#   Abnormal = WY2026  (155.0 mm; retained as the separately designated abnormal year)
#
# Column order follows Low -> Typical -> High -> Abnormal.
SELECTED_CASES = [
    ("Low", 2018),
    ("Typical", 2022),
    ("High", 2019),
    ("Abnormal", 2026),
]

ANALYSIS_YEARS = list(range(2018, 2027))

# -------------------------------------------------------------------------
# East River Plotting Requirements — JHM/AMS v4.
# The guide supersedes the older Arial setting in the v0.8.2 YAML.
# -------------------------------------------------------------------------
FONT_NAME = "Nimbus Sans"
BASE_FONT_SIZE = 18
PANEL_TITLE_FONTSIZE = 20
PANEL_LABEL_FONTSIZE = 18
LINEWIDTH = 2.5
AXIS_LINEWIDTH = 1.0
TICK_WIDTH = 1.0
TICK_LENGTH = 5
EXPORT_DPI = 600

# Existing station-comparison envelope presentation.
ENVELOPE_ALPHA = 0.18
ENVELOPE_LINEWIDTH = 0.2

# ASO observation symbol: match the established plotting convention exactly.
ASO_MARKER = "*"
ASO_SIZE = 230
ASO_FACECOLOR = "#F4C542"
ASO_EDGECOLOR = "black"
ASO_LINEWIDTH = 1.0
ASO_ZORDER = 8

# Four-column adaptation of the established East River development geometry.
FIGSIZE = (28, 16.5)
TOP = 0.95
BOTTOM = 0.15
LEFT = 0.07
HSPACE = 0.30
WSPACE = 0.16
SUPXLABEL_Y = 0.095
SUPYLABEL_X = 0.015

# Variable-specific y-axis rounding.
Y_STEP_MM = {
    "SWE": 50.0,
    "SD": 100.0,
}

# Figure filename stems. Output directories are derived from cfg.output_dir
# after the validated configuration is loaded.
FIGURE_STEMS = {
    "SWE": "spatial_comparison_swe_four_case_years_3x4",
    "SD": "spatial_comparison_snow_depth_four_case_years_3x4",
}


## 3. Helper functions

In [ ]:
def configure_publication_matplotlib() -> str:
    """Apply the current East River publication standard and verify Nimbus Sans."""
    available_names = {f.name for f in fm.fontManager.ttflist}
    if FONT_NAME not in available_names:
        raise RuntimeError(
            "Nimbus Sans is required by East_River_Plotting_Requirements_JHM_AMS_v4.md "
            "but is not available in this Python environment. Install/enable Nimbus Sans "
            "before producing final publication figures; do not accept a silent fallback."
        )

    mpl.rcParams.update({
        "font.family": FONT_NAME,
        "font.size": BASE_FONT_SIZE,
        "axes.titlesize": PANEL_TITLE_FONTSIZE,
        "axes.titleweight": "normal",
        "axes.labelweight": "normal",
        "font.weight": "normal",
        "axes.linewidth": AXIS_LINEWIDTH,
        "xtick.major.width": TICK_WIDTH,
        "ytick.major.width": TICK_WIDTH,
        "xtick.major.size": TICK_LENGTH,
        "ytick.major.size": TICK_LENGTH,
        "legend.frameon": True,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })
    return FONT_NAME


def require_columns(df: pd.DataFrame, required: set[str], name: str) -> None:
    missing = sorted(required.difference(df.columns))
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")


def nice_ymax(values, step: float) -> float:
    """Round the full finite maximum upward with 5% headroom."""
    arrays = []
    for value in values:
        arr = np.asarray(value, dtype=float).ravel()
        arr = arr[np.isfinite(arr)]
        if arr.size:
            arrays.append(arr)

    if not arrays:
        return float(step)

    top = float(np.nanmax(np.concatenate(arrays))) * 1.05
    return float(np.ceil(top / step) * step)


def format_water_year_axis(ax: plt.Axes, wy: int) -> None:
    """Format a full Oct-Sep water-year axis with two-month major ticks."""
    start, end = water_year_bounds(int(wy))
    ax.set_xlim(start, end)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(AXIS_LINEWIDTH)
    ax.spines["bottom"].set_linewidth(AXIS_LINEWIDTH)
    ax.tick_params(
        axis="both",
        which="major",
        width=TICK_WIDTH,
        length=TICK_LENGTH,
        direction="out",
    )


def _parse_aso_filename(path: Path) -> tuple[str, pd.Timestamp] | None:
    """Parse the ASO repository/HPC naming convention."""
    match = re.match(
        r"^ASO_50M_(SWE|SD)_USCOGE_(\d{8})\.tif$",
        path.name,
        flags=re.IGNORECASE,
    )
    if match is None:
        return None
    variable = match.group(1).upper()
    date = pd.to_datetime(match.group(2), format="%Y%m%d")
    return variable, date


def discover_raw_aso_files(aso_directory: Path) -> pd.DataFrame:
    """
    Discover every raw ASO SWE/SD GeoTIFF in the configured HPC ASO directory.

    This intentionally does NOT use cfg['aso']['dates']; the expanded HPC archive
    contains additional flight years. Only files matching the repository naming
    convention are considered.
    """
    aso_directory = Path(aso_directory)
    if not aso_directory.is_dir():
        raise FileNotFoundError(f"Configured ASO directory does not exist: {aso_directory}")

    records = []
    for path in sorted(aso_directory.glob("ASO_50M_*_USCOGE_*.tif")):
        parsed = _parse_aso_filename(path)
        if parsed is None:
            continue
        variable, date = parsed
        wy = int(water_year(date))
        if wy not in ANALYSIS_YEARS:
            continue
        records.append({
            "date": date,
            "water_year": wy,
            "variable": variable,
            "path": path,
        })

    manifest = pd.DataFrame(records)
    if manifest.empty:
        raise FileNotFoundError(
            f"No raw ASO SWE/SD GeoTIFFs matching the expected naming convention were found in {aso_directory}"
        )

    duplicate = manifest.duplicated(["date", "variable"], keep=False)
    if duplicate.any():
        dup = manifest.loc[duplicate, ["date", "variable", "path"]]
        raise ValueError(f"Duplicate ASO files for the same date/variable:\n{dup.to_string(index=False)}")

    # Require SWE and SD as a matched pair for each flight date.
    check = manifest.pivot(index="date", columns="variable", values="path")
    missing = check[check.reindex(columns=["SWE", "SD"]).isna().any(axis=1)]
    if len(missing):
        raise FileNotFoundError(
            "ASO flight dates must have both SWE and SD rasters for this notebook. "
            f"Incomplete dates:\n{missing.to_string()}"
        )

    return manifest.sort_values(["date", "variable"]).reset_index(drop=True)


def build_station_metadata(cfg) -> pd.DataFrame:
    """Get the selected SNOTEL station coordinates from the validated project config."""
    configured = {
        int(item["id"]): item
        for item in cfg.section("snotel")["stations"]
    }

    rows = []
    for station_id, station_name in STATIONS:
        if station_id not in configured:
            raise KeyError(f"Station ID {station_id} is absent from the configured SNOTEL station list.")
        item = configured[station_id]
        rows.append({
            "station_id": station_id,
            "station_name": station_name,
            "latitude": float(item["latitude"]),
            "longitude": float(item["longitude"]),
        })
    return pd.DataFrame(rows)


def sample_all_raw_aso_nearest_pixels(
    aso_manifest: pd.DataFrame,
    station_metadata: pd.DataFrame,
) -> pd.DataFrame:
    """
    Sample the single nearest raw ASO raster pixel center at every selected station.

    IMPORTANT: valid_mask=None is deliberate. We select the nearest raster pixel by
    geometry first; if that raw pixel is source nodata, its value remains NaN. We do
    not jump to a farther valid pixel, and we do not invoke the ASO QC workflow.
    """
    lons = station_metadata["longitude"].to_numpy(dtype=float)
    lats = station_metadata["latitude"].to_numpy(dtype=float)

    records = []
    for row in aso_manifest.itertuples(index=False):
        raw_mm, grid, metadata = read_aso_raster(row.path)
        lookups = nearest_grid_cells(
            grid,
            station_longitudes=lons,
            station_latitudes=lats,
            count=1,
            valid_mask=None,
        )

        for station_row, lookup in zip(station_metadata.itertuples(index=False), lookups):
            rr = int(lookup["rows"][0])
            cc = int(lookup["cols"][0])
            distance_m = float(lookup["distance_m"][0])
            value_mm = float(raw_mm[rr, cc]) if np.isfinite(raw_mm[rr, cc]) else np.nan

            records.append({
                "date": pd.Timestamp(row.date),
                "water_year": int(row.water_year),
                "station_id": int(station_row.station_id),
                "station_name": station_row.station_name,
                "variable": row.variable,
                "aso_raw_mm": value_mm,
                "aso_pixel_row": rr,
                "aso_pixel_col": cc,
                "aso_pixel_distance_m": distance_m,
                "aso_file": str(row.path),
                "aso_crs": metadata["crs"],
            })

    return pd.DataFrame(records).sort_values(
        ["date", "variable", "station_id"]
    ).reset_index(drop=True)


def scalar_at_date(frame: pd.DataFrame, station_id: int, date: pd.Timestamp, column: str) -> float:
    """Return one station/date value; missing rows remain NaN and duplicates are rejected."""
    sub = frame[
        frame["station_id"].eq(station_id)
        & frame["date"].eq(pd.Timestamp(date))
    ]
    if len(sub) > 1:
        raise ValueError(
            f"Duplicate rows for station {station_id}, date {pd.Timestamp(date).date()}, column {column}."
        )
    if sub.empty:
        return np.nan
    value = pd.to_numeric(sub.iloc[0][column], errors="coerce")
    return float(value) if pd.notna(value) else np.nan


## 4. Load validated station tables and enforce the new plotting font

In [ ]:
cfg = load_config(CONFIG_PATH)
font_used = configure_publication_matplotlib()

FIGURE_DIR = cfg.output_dir / "figures" / "publication"
TABLE_DIR = cfg.output_dir / "tables" / "publication"
SNOTEL_STATS_CSV = TABLE_DIR / "spatial_comparison_snotel_r_mb_wy2018_2026.csv"
ASO_DIFF_CSV = TABLE_DIR / "spatial_comparison_aso_absolute_differences_all_dates.csv"

table_dir = cfg.output_dir / "tables"

snotel_path = table_dir / "snotel_daily.csv"
isnobal_path = table_dir / "isnobal_station_daily.csv"
noah_path = table_dir / "noahmp_station_daily.csv"

snotel = pd.read_csv(snotel_path, parse_dates=["date"])
isnobal = pd.read_csv(isnobal_path, parse_dates=["date"])
noah = pd.read_csv(noah_path, parse_dates=["date"])

for frame in (snotel, isnobal, noah):
    frame["station_id"] = pd.to_numeric(frame["station_id"], errors="coerce").astype("Int64")
    frame["water_year"] = pd.to_numeric(frame["water_year"], errors="coerce").astype("Int64")
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()

require_columns(
    snotel,
    {"date", "station_id", "water_year", "swe_mm", "snow_depth_mm"},
    "snotel_daily.csv",
)
require_columns(
    isnobal,
    {
        "date",
        "station_id",
        "water_year",
        "swe_nearest_mm",
        "snow_depth_nearest_mm",
        "swe_in_noah_cell_min_mm",
        "swe_in_noah_cell_max_mm",
        "snow_depth_in_noah_cell_min_mm",
        "snow_depth_in_noah_cell_max_mm",
    },
    "isnobal_station_daily.csv",
)
require_columns(
    noah,
    {"date", "station_id", "water_year", "swe_mm", "snow_depth_mm"},
    "noahmp_station_daily.csv",
)

station_metadata = build_station_metadata(cfg)

print("Publication font:", font_used)
print("SNOTEL:", snotel_path)
print("iSnobal:", isnobal_path)
print("Noah-MP:", noah_path)
print("ASO raw directory:", cfg.data["paths"]["aso_directory"])
print("Rows:", len(snotel), len(isnobal), len(noah))


## 5. Confirm the four watershed-peak-SWE-selected years and SNOTEL coordinates

The four plotted years are the cases selected from the completed watershed annual peak iSnobal SWE ranking:

- **Low:** WY2018
- **Typical:** WY2022
- **High:** WY2019
- **Abnormal:** WY2026

They are used directly here and are **not recomputed from the station-scale SNOTEL record**.


In [ ]:
# -------------------------------------------------------------------------
# Four-year figure design from the watershed annual peak-SWE ranking.
# -------------------------------------------------------------------------
# Selected cases:
#   Low      -> WY2018
#   Typical  -> WY2022
#   High     -> WY2019
#   Abnormal -> WY2026
#
# All WY2018-WY2026 are still retained in the statistics calculations below.

selected_years = [wy for _, wy in SELECTED_CASES]

missing_analysis_years = sorted(set(selected_years).difference(ANALYSIS_YEARS))
if missing_analysis_years:
    raise ValueError(
        f"Selected plot years are outside ANALYSIS_YEARS: {missing_analysis_years}"
    )

print("Watershed peak-SWE-selected four-year set:")
for label, wy in SELECTED_CASES:
    print(f"  {label}: WY{wy}")

print("\nSNOTEL station coordinates read from the validated config:")
print(
    station_metadata[
        ["station_id", "station_name", "latitude", "longitude"]
    ].to_string(index=False)
)


## 6. Discover all raw ASO flights and sample the nearest station pixels

In [ ]:
aso_manifest = discover_raw_aso_files(Path(cfg.data["paths"]["aso_directory"]))
aso_nearest = sample_all_raw_aso_nearest_pixels(aso_manifest, station_metadata)

flight_dates = (
    aso_manifest[["date", "water_year"]]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Discovered {len(flight_dates)} paired ASO flight dates in WY2018-WY2026:")
print(flight_dates.to_string(index=False))
print()
print("Nearest raw ASO station samples:", len(aso_nearest))
print(aso_nearest.head(12).to_string(index=False))


print("\nNearest-pixel distance diagnostic by station (m):")
distance_summary = (
    aso_nearest.groupby(["station_id", "station_name"])["aso_pixel_distance_m"]
    .agg(["min", "median", "max"])
    .round(2)
)
print(distance_summary.to_string())

# Diagnostic only: do not alter the selected raw ASO pixel based on its value.
if np.isfinite(aso_nearest["aso_pixel_distance_m"]).any():
    max_distance = float(np.nanmax(aso_nearest["aso_pixel_distance_m"]))
    if max_distance > 100.0:
        print(
            f"WARNING: maximum SNOTEL-to-ASO pixel-center distance is {max_distance:.1f} m. "
            "Inspect station coordinates / ASO coverage before publication."
        )


## 7. SNOTEL-reference `r` and `MB` for all WY2018-WY2026

`MB` follows the validated workflow sign convention: **model - SNOTEL**. Statistics use paired finite daily values only, via the workflow `pearson_mean_bias()` helper. These statistics are saved to CSV and are **not annotated on the figures**.

In [ ]:
VARIABLE_COLUMNS = {
    "SWE": {
        "label": "SWE",
        "ylabel": "SWE (mm)",
        "snotel": "swe_mm",
        "isnobal": "swe_nearest_mm",
        "noah": "swe_mm",
        "envelope_min": "swe_in_noah_cell_min_mm",
        "envelope_max": "swe_in_noah_cell_max_mm",
    },
    "SD": {
        "label": "Snow depth",
        "ylabel": "Snow depth (mm)",
        "snotel": "snow_depth_mm",
        "isnobal": "snow_depth_nearest_mm",
        "noah": "snow_depth_mm",
        "envelope_min": "snow_depth_in_noah_cell_min_mm",
        "envelope_max": "snow_depth_in_noah_cell_max_mm",
    },
}


def compute_snotel_r_mb_all_years(
    snotel: pd.DataFrame,
    isnobal: pd.DataFrame,
    noah: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for station_id, station_name in STATIONS:
        for wy in ANALYSIS_YEARS:
            for variable, spec in VARIABLE_COLUMNS.items():
                ref = (
                    snotel[
                        snotel["station_id"].eq(station_id)
                        & snotel["water_year"].eq(wy)
                    ][["date", spec["snotel"]]]
                    .rename(columns={spec["snotel"]: "reference"})
                )

                for model_name, model_frame, model_column in [
                    ("iSnobal", isnobal, spec["isnobal"]),
                    ("Noah-MP", noah, spec["noah"]),
                ]:
                    mod = (
                        model_frame[
                            model_frame["station_id"].eq(station_id)
                            & model_frame["water_year"].eq(wy)
                        ][["date", model_column]]
                        .rename(columns={model_column: "model"})
                    )

                    paired = ref.merge(mod, on="date", how="inner", validate="one_to_one")
                    stats = pearson_mean_bias(
                        paired["reference"].to_numpy(dtype=float),
                        paired["model"].to_numpy(dtype=float),
                    )

                    rows.append({
                        "station_id": station_id,
                        "station_name": station_name,
                        "water_year": wy,
                        "variable": variable,
                        "reference": "SNOTEL",
                        "model": model_name,
                        "n": int(stats["n"]),
                        "r": float(stats["pearson_r"]),
                        "MB_mm": float(stats["mean_bias"]),
                    })

    return pd.DataFrame(rows).sort_values(
        ["variable", "station_id", "water_year", "model"]
    ).reset_index(drop=True)


snotel_stats = compute_snotel_r_mb_all_years(snotel, isnobal, noah)

TABLE_DIR.mkdir(parents=True, exist_ok=True)
snotel_stats.to_csv(SNOTEL_STATS_CSV, index=False)

print("Saved:", SNOTEL_STATS_CSV)
print("Rows:", len(snotel_stats))
print(snotel_stats.head(18).to_string(index=False))


## 8. Absolute ASO magnitude differences for every flight date

For each `station × ASO date × SWE/SD`, the notebook compares the nearest **raw** ASO pixel with the same-day SNOTEL, nearest iSnobal, and nearest Noah-MP station values and stores:

\[
|\mathrm{dataset} - \mathrm{ASO}|.
\]

No ASO QC is applied, and these differences are **not plotted as panel text**.

In [ ]:
def build_aso_absolute_difference_table(
    aso_nearest: pd.DataFrame,
    snotel: pd.DataFrame,
    isnobal: pd.DataFrame,
    noah: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for aso in aso_nearest.itertuples(index=False):
        spec = VARIABLE_COLUMNS[aso.variable]
        station_id = int(aso.station_id)
        date = pd.Timestamp(aso.date)
        aso_mm = float(aso.aso_raw_mm) if np.isfinite(aso.aso_raw_mm) else np.nan

        comparisons = [
            ("SNOTEL", scalar_at_date(snotel, station_id, date, spec["snotel"])),
            ("iSnobal", scalar_at_date(isnobal, station_id, date, spec["isnobal"])),
            ("Noah-MP", scalar_at_date(noah, station_id, date, spec["noah"])),
        ]

        for dataset_name, dataset_mm in comparisons:
            absolute_difference_mm = (
                abs(dataset_mm - aso_mm)
                if np.isfinite(dataset_mm) and np.isfinite(aso_mm)
                else np.nan
            )

            rows.append({
                "date": date,
                "water_year": int(aso.water_year),
                "station_id": station_id,
                "station_name": aso.station_name,
                "variable": aso.variable,
                "dataset": dataset_name,
                "dataset_mm": dataset_mm,
                "ASO_mm": aso_mm,
                "absolute_difference_mm": absolute_difference_mm,
                "aso_pixel_row": int(aso.aso_pixel_row),
                "aso_pixel_col": int(aso.aso_pixel_col),
                "aso_pixel_distance_m": float(aso.aso_pixel_distance_m),
                "aso_file": aso.aso_file,
            })

    return pd.DataFrame(rows).sort_values(
        ["date", "variable", "station_id", "dataset"]
    ).reset_index(drop=True)


aso_differences = build_aso_absolute_difference_table(
    aso_nearest=aso_nearest,
    snotel=snotel,
    isnobal=isnobal,
    noah=noah,
)

aso_differences.to_csv(ASO_DIFF_CSV, index=False)

print("Saved:", ASO_DIFF_CSV)
print("Rows:", len(aso_differences))
print(aso_differences.head(18).to_string(index=False))


## 9. Generic four-case-year plotting function for SWE and SD

In [ ]:
def plot_spatial_comparison_four_case_years(
    variable: str,
    snotel: pd.DataFrame,
    isnobal: pd.DataFrame,
    noah: pd.DataFrame,
    aso_nearest: pd.DataFrame,
    show_figure: bool = True,
) -> tuple[Path, Path]:
    """Create one 3x4 station comparison figure for SWE or snow depth."""
    if variable not in VARIABLE_COLUMNS:
        raise ValueError(f"variable must be one of {list(VARIABLE_COLUMNS)}, got {variable!r}")

    spec = VARIABLE_COLUMNS[variable]
    selected_years = [wy for _, wy in SELECTED_CASES]
    nrows = len(STATIONS)
    ncols = len(SELECTED_CASES)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=FIGSIZE,
        sharey="row",
        squeeze=False,
    )

    # One y-axis range per station row, shared across all four plotted years.
    # The full raw ASO values are included in the limit calculation (no clipping/QC).
    row_ymax: dict[int, float] = {}
    for station_id, _station_name in STATIONS:
        ss = snotel[
            snotel["station_id"].eq(station_id)
            & snotel["water_year"].isin(selected_years)
        ]
        ii = isnobal[
            isnobal["station_id"].eq(station_id)
            & isnobal["water_year"].isin(selected_years)
        ]
        nn = noah[
            noah["station_id"].eq(station_id)
            & noah["water_year"].isin(selected_years)
        ]
        aa = aso_nearest[
            aso_nearest["station_id"].eq(station_id)
            & aso_nearest["water_year"].isin(selected_years)
            & aso_nearest["variable"].eq(variable)
        ]

        row_ymax[station_id] = nice_ymax(
            [
                ss[spec["snotel"]],
                ii[spec["envelope_max"]],
                nn[spec["noah"]],
                aa["aso_raw_mm"],
            ],
            step=Y_STEP_MM[variable],
        )

    for row, (station_id, station_name) in enumerate(STATIONS):
        for col, (case_label, wy) in enumerate(SELECTED_CASES):
            ax = axes[row, col]

            ss = (
                snotel[
                    snotel["station_id"].eq(station_id)
                    & snotel["water_year"].eq(wy)
                ]
                .sort_values("date")
                .copy()
            )
            ii = (
                isnobal[
                    isnobal["station_id"].eq(station_id)
                    & isnobal["water_year"].eq(wy)
                ]
                .sort_values("date")
                .copy()
            )
            nn = (
                noah[
                    noah["station_id"].eq(station_id)
                    & noah["water_year"].eq(wy)
                ]
                .sort_values("date")
                .copy()
            )
            aa = (
                aso_nearest[
                    aso_nearest["station_id"].eq(station_id)
                    & aso_nearest["water_year"].eq(wy)
                    & aso_nearest["variable"].eq(variable)
                ]
                .sort_values("date")
                .copy()
            )

            if ss.empty or ii.empty or nn.empty:
                raise ValueError(
                    f"Missing station time series for {station_name} (ID {station_id}), WY{wy}, {variable}."
                )

            # Existing iSnobal spatial envelope inside the station's nearest Noah-MP cell.
            ax.fill_between(
                ii["date"],
                ii[spec["envelope_min"]],
                ii[spec["envelope_max"]],
                facecolor=MODEL_COLORS["iSnobal"],
                edgecolor=MODEL_COLORS["iSnobal"],
                alpha=ENVELOPE_ALPHA,
                linewidth=ENVELOPE_LINEWIDTH,
                zorder=1,
            )

            # Dataset curves. Legend wording is dataset name only.
            ax.plot(
                ii["date"],
                ii[spec["isnobal"]],
                color=MODEL_COLORS["iSnobal"],
                lw=LINEWIDTH,
                zorder=2,
            )
            ax.plot(
                nn["date"],
                nn[spec["noah"]],
                color=MODEL_COLORS["Noah-MP"],
                lw=LINEWIDTH,
                zorder=3,
            )
            ax.plot(
                ss["date"],
                ss[spec["snotel"]],
                color=MODEL_COLORS["SNOTEL"],
                lw=LINEWIDTH,
                zorder=4,
            )

            # All ASO flight observations available in this plotted water year.
            # Match the established ASO symbol convention exactly.
            if not aa.empty:
                ax.scatter(
                    aa["date"],
                    aa["aso_raw_mm"],
                    marker=ASO_MARKER,
                    s=ASO_SIZE,
                    facecolors=ASO_FACECOLOR,
                    edgecolors=ASO_EDGECOLOR,
                    linewidths=ASO_LINEWIDTH,
                    zorder=ASO_ZORDER,
                )

            ax.set_ylim(0, row_ymax[station_id])
            format_water_year_axis(ax, wy)

            # Sequential panel labels in row-major order:
            # first station row (a)-(d), second (e)-(h), third (i)-(l).
            panel_index = row * ncols + col
            panel_letter = chr(ord("a") + panel_index)
            ax.text(
                0.01,
                1.02,
                f"({panel_letter})",
                transform=ax.transAxes,
                ha="left",
                va="bottom",
                fontsize=PANEL_LABEL_FONTSIZE,
                fontweight="normal",
                clip_on=False,
            )

            if row == 0:
                ax.set_title(
                    f"{case_label} (WY {wy})",
                    fontsize=PANEL_TITLE_FONTSIZE,
                    fontweight="normal",
                    loc="center",
                    pad=8,
                )

            if col == 0:
                ax.text(
                    0.97,
                    0.95,
                    station_name,
                    transform=ax.transAxes,
                    ha="right",
                    va="top",
                    fontsize=BASE_FONT_SIZE,
                    fontweight="normal",
                )

    fig.supxlabel("Month", y=SUPXLABEL_Y)
    fig.supylabel(spec["ylabel"], x=SUPYLABEL_X)

    # Dataset names only; the iSnobal envelope is not a separate legend item.
    axes[0, 0].legend(
        handles=[
            Line2D([], [], color=MODEL_COLORS["SNOTEL"], lw=LINEWIDTH, label="SNOTEL"),
            Line2D([], [], color=MODEL_COLORS["iSnobal"], lw=LINEWIDTH, label="iSnobal"),
            Line2D([], [], color=MODEL_COLORS["Noah-MP"], lw=LINEWIDTH, label="Noah-MP"),
            axes[0, 0].scatter(
                [], [],
                marker=ASO_MARKER,
                s=ASO_SIZE,
                facecolors=ASO_FACECOLOR,
                edgecolors=ASO_EDGECOLOR,
                linewidths=ASO_LINEWIDTH,
                label="ASO",
            ),
        ],
        loc="upper left",
        ncol=1,
        frameon=True,
        facecolor="white",
        edgecolor="0.4",
        framealpha=0.9,
        fancybox=False,
        borderpad=0.5,
        handlelength=2.2,
    )

    fig.subplots_adjust(
        top=TOP,
        bottom=BOTTOM,
        left=LEFT,
        hspace=HSPACE,
        wspace=WSPACE,
    )

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    stem = FIGURE_STEMS[variable]
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"

    fig.savefig(
        png_path,
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        pad_inches=0.05,
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    if show_figure:
        plt.show()
    else:
        plt.close(fig)

    return png_path, pdf_path


## 10. Generate SWE and snow-depth figures

In [ ]:
swe_png, swe_pdf = plot_spatial_comparison_four_case_years(
    variable="SWE",
    snotel=snotel,
    isnobal=isnobal,
    noah=noah,
    aso_nearest=aso_nearest,
    show_figure=True,
)

sd_png, sd_pdf = plot_spatial_comparison_four_case_years(
    variable="SD",
    snotel=snotel,
    isnobal=isnobal,
    noah=noah,
    aso_nearest=aso_nearest,
    show_figure=True,
)

print("Saved SWE PNG:", swe_png)
print("Saved SWE PDF:", swe_pdf)
print("Saved SD PNG:", sd_png)
print("Saved SD PDF:", sd_pdf)


## Outputs

The notebook writes the following products on the HPC.

**Figures — four watershed peak-SWE-selected case years**

- `output/output1/figures/publication/spatial_comparison_swe_four_case_years_3x4.png`
- `output/output1/figures/publication/spatial_comparison_swe_four_case_years_3x4.pdf`
- `output/output1/figures/publication/spatial_comparison_snow_depth_four_case_years_3x4.png`
- `output/output1/figures/publication/spatial_comparison_snow_depth_four_case_years_3x4.pdf`

Columns are:

1. **Low — WY2018**
2. **Typical — WY2022**
3. **High — WY2019**
4. **Abnormal — WY2026**

Panels are labeled sequentially in row-major order:

- Butte: **(a)–(d)**
- Schofield Pass: **(e)–(h)**
- Upper Taylor: **(i)–(l)**

**Statistics — all WY2018-WY2026 / all available ASO dates**

- `output/output1/tables/publication/spatial_comparison_snotel_r_mb_wy2018_2026.csv`  
  Daily paired `r` and `MB = model - SNOTEL` for iSnobal and Noah-MP, by station, water year, and variable.

- `output/output1/tables/publication/spatial_comparison_aso_absolute_differences_all_dates.csv`  
  Raw nearest-pixel ASO value, same-day dataset value, and `|dataset - ASO|` for SNOTEL, iSnobal, and Noah-MP, by station, flight date, and variable.

The plotted panels contain **no statistical text annotations** and use legend labels **SNOTEL / iSnobal / Noah-MP / ASO** only.
